# Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"


from crispr_millipede import encoding as cme
from crispr_millipede import modelling as cmm

In [3]:
import pandas as pd
import numpy as np

from Bio import SeqIO
from Bio.Seq import Seq

import matplotlib.pyplot as plt

In [5]:
amplicon = "ACATGCTCTAGTGAAAGCCAGTCTGGGCAGCTGGGTAGCTAATGAGGGGATTAGAGAGATTTTGTTGAATGAAAGGCAGATTGAGTCCTGCTACTCGCCCCCTTCATTCCCCTTCATTCATGCCTCATTCTTCCGCCTCCCAGCCGCCTCAACTGGCCAAAGGGAAGTGGAGGCCCTGCCACCTGTAGGGAGGGTCCCCTGGGGCTTGCCCACAGCAAACAGGAAGTCACAGCCTGGTGAGATGGGCCTGGGAATCAGCCACTGAGAAAGTGGGTCTCTTGGGTCCCTGAATTCTTTTTCTGAGTCCCTGCAGCAGTGAAAAAGACACAGAGGCACATAGAGAGTG"
print(f"Length of amplicon {len(amplicon)}")

Length of amplicon 346


# Production

## Encoding

In [6]:

design_matrix_spec = cmm.MillipedeDesignMatrixProcessingSpecification(
    wt_normalization=False,
    total_normalization=True,
    sigma_scale_normalized=True,
    decay_sigma_scale=True,
    K_enriched=5,
    K_baseline=5,
    a_parameter=0.0001, #0.0001
    set_offset_as_presort = True,
    offset_normalized = False,
    offset_psuedocount = 1 # OTHERWISE will get infinity error when taking log(count +1) for offset, since 0 counts could be acceptable depending on thresholding
)

millipede_model_specification_set = {
    "joint_replicate_per_experiment_models" : cmm.MillipedeModelSpecification(
        model_types=[cmm.MillipedeModelType.NORMAL_SIGMA_SCALED, cmm.MillipedeModelType.NORMAL],
        replicate_merge_strategy=cmm.MillipedeReplicateMergeStrategy.COVARIATE,
        experiment_merge_strategy=cmm.MillipedeExperimentMergeStrategy.SEPARATE,
        S = 5,
        tau = 0.01,
        tau_intercept = 0.0001,
        cutoff_specification=cmm.MillipedeCutoffSpecification(
            per_replicate_each_condition_num_cutoff = 0, 
            per_replicate_all_condition_num_cutoff = 1, 
            all_replicate_num_cutoff = 0, 
            all_experiment_num_cutoff = 0,
            baseline_pop_all_condition_each_replicate_num_cutoff = 3,
            baseline_pop_all_condition_acceptable_rep_count = 2,
            enriched_pop_all_condition_each_replicate_num_cutoff = 3,
            enriched_pop_all_condition_acceptable_rep_count = 2,
            presort_pop_all_condition_each_replicate_num_cutoff = 3,
            presort_pop_all_condition_acceptable_rep_count = 2,
            
        ),
        design_matrix_processing_specification=design_matrix_spec
    )
}

In [7]:
paired_end_experiments_inputdata_denoised = cmm.MillipedeInputDataExperimentalGroup(
    data_directory="./", 
    enriched_pop_fn_experiment_list = ["20240905_v0_1_89_sg219_encoding_dataframes_variantdenoised_removed_CRISPResso2Merged_trimmed_ABE8e_encodings_rep{}.tsv"],
    enriched_pop_df_reads_colname = "#Reads_CD19minus",
    baseline_pop_fn_experiment_list = ["20240905_v0_1_89_sg219_encoding_dataframes_variantdenoised_removed_CRISPResso2Merged_trimmed_ABE8e_encodings_rep{}.tsv"],
    baseline_pop_df_reads_colname = "#Reads_CD19plus", 
    presort_pop_fn_experiment_list = ["20240905_v0_1_89_sg219_encoding_dataframes_variantdenoised_removed_CRISPResso2Merged_trimmed_ABE8e_encodings_rep{}.tsv"],
    presort_pop_df_reads_colname = '#Reads_presort',
    experiment_labels = ["EvoCDA"],
    reps = [0,1,2],
    millipede_model_specification_set = millipede_model_specification_set
   )


Performing initial input validation checks...
Passed validation.
Retrieving data for
	Replicate Merge Strategy: MillipedeReplicateMergeStrategy.COVARIATE 
	Experiment Merge Strategy MillipedeExperimentMergeStrategy.SEPARATE
	Cutoff: 
                    per_replicate_each_condition_num_cutoff=0, 
                    per_replicate_presort_condition_num_cutoff=0, 
                    per_replicate_all_condition_num_cutoff=1, 
                    all_replicate_num_cutoff=0, 
                    all_experiment_num_cutoff=0, 
                    baseline_pop_per_condition_acceptable_rep_count=0, 
                    baseline_pop_per_condition_acceptable_rep_count=0, 
                    baseline_pop_per_condition_acceptable_rep_count=0,
                    enriched_pop_per_condition_each_replicate_num_cutoff=0,
                    enriched_pop_per_condition_acceptable_rep_count=0,
                    presort_pop_per_condition_each_replicate_num_cutoff=0,
                    presort_pop_per_

## Count Table

In [8]:
data = paired_end_experiments_inputdata_denoised.millipede_model_specification_set_with_data['joint_replicate_per_experiment_models'][1].data[0]
alleles = [data[data['intercept_exp0_rep0'] == 1], data[data['intercept_exp0_rep1'] == 1], data[data['intercept_exp0_rep2'] == 1]]

In [9]:
allele_tables = []

for rep in range(3):
    cols = list(alleles[rep].columns.values)
    restricted_cols = [col for col in cols if ">" in col]
    
    label = []
    edits = []
    
    minus = []
    plus = []
    presort = []    

    for index, row in alleles[rep].iterrows():
        row_restricted = row[restricted_cols]
        cols_selected = row_restricted[row_restricted == 1].index.tolist()
        
        allele = amplicon
        for col in cols_selected:
            demarker = col.find(">")
            pos = int(col[:demarker - 1])
            alt = col[demarker+1:]
            allele = allele[:pos] + alt + allele[pos + 1:]
                 
        if len(cols_selected) == 0:
            edits.append("wt")
        else:
            edits.append(",".join(cols_selected))
        
        minus.append(row["#Reads_CD19minus_raw"])
        plus.append(row["#Reads_CD19plus_raw"])
        
    dicto = {"edits": edits,
             "CD19minus_rep" + str(rep + 1): minus,
             "CD19plus_rep" + str(rep + 1): plus}
    
    allele_tables.append(pd.DataFrame(dicto).set_index("edits"))

In [10]:
#Drop columns with insufficient read count

'''
for table in allele_tables:
    for index, row in table.iterrows():
        total_count = row["CD19minus"] + row["CD19plus"] + row["presort"]
        if total_count < 10:
            table.drop(index, inplace = True)
'''

'\nfor table in allele_tables:\n    for index, row in table.iterrows():\n        total_count = row["CD19minus"] + row["CD19plus"] + row["presort"]\n        if total_count < 10:\n            table.drop(index, inplace = True)\n'

In [11]:
merged_df = pd.merge(allele_tables[0], allele_tables[1], on='edits', how='outer').fillna(0)
merged_df = pd.merge(merged_df, allele_tables[2], on='edits', how='outer').fillna(0)

In [12]:
count_df = merged_df.T
count_df.index.names = ['sample']

In [16]:
count_df

edits,wt,324A>G,322A>G,321A>G,320A>G,319A>G,318A>G,316T>C,314A>G,311A>G,...,"28A>G,220A>G,223A>G,224A>G,228A>G","28A>G,220A>G,223A>G,224A>G,228A>G,230A>G",23T>C,"23T>C,220A>G,223A>G,224A>G,228A>G",21T>C,"21T>C,220A>G,223A>G,224A>G,228A>G","21T>C,220A>G,223A>G,224A>G,228A>G,230A>G","223A>G,287T>C","217A>G,218A>G,220A>G,223A>G,224A>G","131T>C,220A>G,223A>G,224A>G,228A>G,230A>G"
sample,,,,,,,,,,,,,,,,,,,,,
CD19minus_rep1,11559.0,2.0,20.0,0.0,0.0,4.0,0.0,10.0,3.0,6.0,...,3.0,2.0,5.0,4.0,4.0,10.0,1.0,0.0,0.0,0.0
CD19plus_rep1,24478.0,7.0,15.0,5.0,1.0,6.0,4.0,5.0,2.0,7.0,...,3.0,2.0,7.0,1.0,11.0,5.0,1.0,0.0,0.0,0.0
CD19minus_rep2,7915.0,1.0,3.0,3.0,1.0,1.0,0.0,5.0,2.0,3.0,...,4.0,1.0,6.0,5.0,4.0,11.0,3.0,0.0,3.0,3.0
CD19plus_rep2,24824.0,8.0,20.0,3.0,4.0,4.0,4.0,10.0,5.0,8.0,...,4.0,1.0,9.0,4.0,17.0,5.0,2.0,1.0,0.0,1.0
CD19minus_rep3,7824.0,2.0,4.0,1.0,1.0,4.0,0.0,3.0,3.0,3.0,...,9.0,1.0,1.0,5.0,3.0,10.0,3.0,1.0,27.0,3.0
CD19plus_rep3,19548.0,7.0,10.0,2.0,6.0,6.0,4.0,6.0,4.0,2.0,...,7.0,2.0,3.0,4.0,7.0,8.0,2.0,1.0,0.0,2.0


In [20]:
count_df.to_csv("ABEsg218CountDf.tsv", sep="\t")

## Metadata Table


In [14]:
samples = list(count_df.index)
condition = ["CD19-", "CD19+"] * 3
replicate = [1, 1, 2, 2, 3, 3]
metadata_dict = {"sample": samples,
                 "condition": condition,
                 "replicate": replicate}

metadata_df = pd.DataFrame(metadata_dict).set_index("sample")

In [21]:
metadata_df.to_csv("ABEsg218MetadataDf.tsv", sep="\t")